# Mathematical Transformers - Kaggle Multi-GPU (T4 x 2) Training Notebook

This notebook trains a Pre-LN Decoder-Only Transformer on symbolic arithmetic and mathematical function evaluations with adaptive curriculum data generation.

### Key Features:
- **2-GPU Data Parallel Training**: Uses JAX `pmap` (`jax.lax.pmean`) to train across both Kaggle T4 GPUs simultaneously.
- **On-The-Go Live Plotting**: Real-time visualization of training loss, validation loss, and validation accuracy inside Jupyter.
- **Debug Mode**: Top-level `DEBUG_MODE = True` toggle to test data generation, training, plotting, validation, and saving all at once in ~1 minute.
- **Dataset & Model Export**: Automatically saves model checkpoints (Orbax) and generated dataset shards (`.jsonl`) to `/kaggle/working/`.

In [ ]:
# Cell 1: Setup & Environment
import os
import sys
import subprocess

# Ensure repository is available and in path
if not os.path.exists("src"):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/USER/math-transformers.git"], check=True)
    os.chdir("math-transformers")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print("Current Working Directory:", os.getcwd())

# Check JAX device availability
import jax
devices = jax.devices()
print(f"JAX Devices Detected ({len(devices)}): {devices}")

In [ ]:
# Cell 2: Configuration & Debug Mode Toggle
# ==========================================
# DEBUG MODE TOGGLE
# ==========================================
# Set DEBUG_MODE = True to run a quick 1-minute dry run testing all features at once.
# Set DEBUG_MODE = False for full GPU training on Kaggle T4*2.
DEBUG_MODE = True

if DEBUG_MODE:
    print("=== RUNNING IN DEBUG MODE (Fast 1-minute full pipeline test) ===")
    config = {
        "seed": 42,
        "precision": "float32",
        "model": {
            "context_len": 32,
            "num_layers": 2,
            "num_heads": 2,
            "emb_dim": 64,
            "mlp_dim": 128,
            "pos_emb_type": "learned"
        },
        "data": {
            "max_depth": 2,
            "float_precision": 1,
            "enabled_ops": ["+", "-", "*", "/", "sin", "cos"],
            "val_size": 50,
            "test_size": 50
        },
        "training": {
            "total_steps": 100,
            "warmup_steps": 10,
            "learning_rate": 0.001,
            "batch_size": 16,
            "max_grad_norm": 1.0,
            "weight_decay": 0.01,
            "eval_interval": 20,
            "save_interval": 50,
            "log_interval": 5,
            "checkpoint_dir": "./checkpoints_debug",
            "accuracy_epsilon": 0.05
        },
        "curriculum": {
            "enabled": True,
            "ema_alpha": 0.2,
            "temperature": 1.0,
            "floor_prob": 0.01,
            "overfit_ratio": 3.0,
            "overfit_train_threshold": 0.3,
            "overfit_decay": 0.1,
            "update_interval": 20
        }
    }
else:
    print("=== RUNNING IN PRODUCTION MODE (Kaggle T4*2 Multi-GPU Training) ===")
    config = {
        "seed": 42,
        "precision": "bfloat16",
        "model": {
            "context_len": 64,
            "num_layers": 6,
            "num_heads": 8,
            "emb_dim": 256,
            "mlp_dim": 1024,
            "pos_emb_type": "sinusoidal"
        },
        "data": {
            "max_depth": 3,
            "float_precision": 1,
            "enabled_ops": ["+", "-", "*", "/", "^", "sin", "cos", "tan", "log", "ln", "exp", "sqrt", "abs"],
            "val_size": 1000,
            "test_size": 1000
        },
        "training": {
            "total_steps": 25000,
            "warmup_steps": 1000,
            "learning_rate": 0.0003,
            "batch_size": 128,
            "max_grad_norm": 1.0,
            "weight_decay": 0.01,
            "eval_interval": 500,
            "save_interval": 2500,
            "log_interval": 50,
            "checkpoint_dir": "/kaggle/working/checkpoints",
            "accuracy_epsilon": 0.01
        },
        "curriculum": {
            "enabled": True,
            "ema_alpha": 0.1,
            "temperature": 0.5,
            "floor_prob": 0.005,
            "overfit_ratio": 3.0,
            "overfit_train_threshold": 0.15,
            "overfit_decay": 0.1,
            "update_interval": 250
        }
    }

In [ ]:
# Cell 3: On-The-Go Live Plotting Utility
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

class LivePlotter:
    """
    Renders live, dynamic training and validation metrics plots inside Jupyter notebook cells.
    """
    def __init__(self):
        self.train_steps = []
        self.train_losses = []
        self.val_steps = []
        self.val_losses = []
        self.val_exact_matches = []
        self.val_tolerant_accs = []

    def update_train(self, step: int, loss: float):
        self.train_steps.append(step)
        self.train_losses.append(loss)

    def update_val(self, step: int, loss: float, exact_match: float, tolerant_acc: float):
        self.val_steps.append(step)
        self.val_losses.append(loss)
        self.val_exact_matches.append(exact_match * 100.0)
        self.val_tolerant_accs.append(tolerant_acc * 100.0)

    def plot(self):
        try:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            
            # Subplot 1: Losses
            if self.train_steps:
                ax1.plot(self.train_steps, self.train_losses, label="Train Loss", color="dodgerblue", alpha=0.8, linewidth=1.5)
            if self.val_steps:
                ax1.plot(self.val_steps, self.val_losses, label="Val Loss", color="crimson", marker="o", linewidth=2.0)
            ax1.set_xlabel("Step")
            ax1.set_ylabel("Cross Entropy Loss")
            ax1.set_title("Training & Validation Loss (On The Go)")
            ax1.grid(True, linestyle="--", alpha=0.5)
            ax1.legend()

            # Subplot 2: Accuracy
            if self.val_steps:
                ax2.plot(self.val_steps, self.val_exact_matches, label="Exact Match %", color="green", marker="s", linewidth=2.0)
                ax2.plot(self.val_steps, self.val_tolerant_accs, label="Tolerant Acc %", color="darkorange", marker="^", linewidth=2.0)
            ax2.set_xlabel("Step")
            ax2.set_ylabel("Accuracy (%)")
            ax2.set_title("Validation Accuracy Metrics")
            ax2.set_ylim(-5, 105)
            ax2.grid(True, linestyle="--", alpha=0.5)
            ax2.legend()

            plt.tight_layout()
            display(plt.gcf())
            plt.close(fig)
        except Exception:
            pass

In [ ]:
# Cell 4: Data Sampler, Curriculum, Model & Multi-GPU Setup
import time
import json
import numpy as np
import jax.numpy as jnp
import optax
import flax.jax_utils as jutils

from src.tokenizer.tokenizer import Tokenizer
from src.data.sampler import ExpressionSampler
from src.data.curriculum import CurriculumTracker
from src.model.transformer import TransformerDecoder
from src.train import CheckpointManager, CustomTrainState, train_step, make_parallel_train_step
from src.eval import evaluate_on_dataset

# Initialize Tokenizer and Sampler
tokenizer = Tokenizer()
seed = config.get("seed", 42)
np.random.seed(seed)

sampler = ExpressionSampler(
    tokenizer=tokenizer,
    max_depth=config["data"]["max_depth"],
    float_precision=config["data"]["float_precision"],
    context_len=config["model"]["context_len"],
    seed=seed,
    enabled_ops=config["data"]["enabled_ops"],
    val_size=config["data"]["val_size"],
    test_size=config["data"]["test_size"]
)

# Curriculum Tracker Setup
cur_cfg = config.get("curriculum", {})
cur_tracker = CurriculumTracker(
    categories=sampler.categories,
    enabled=cur_cfg.get("enabled", True),
    ema_alpha=cur_cfg.get("ema_alpha", 0.2),
    temperature=cur_cfg.get("temperature", 1.0),
    floor_prob=cur_cfg.get("floor_prob", 0.01),
    overfit_ratio=cur_cfg.get("overfit_ratio", 3.0),
    overfit_train_threshold=cur_cfg.get("overfit_train_threshold", 0.3),
    overfit_decay=cur_cfg.get("overfit_decay", 0.1),
    update_interval=cur_cfg.get("update_interval", 100)
)

# Model Initialization
compute_dtype = jnp.bfloat16 if config.get("precision") == "bfloat16" else jnp.float32

model = TransformerDecoder(
    vocab_size=tokenizer.vocab_size,
    context_len=config["model"]["context_len"],
    num_layers=config["model"]["num_layers"],
    num_heads=config["model"]["num_heads"],
    emb_dim=config["model"]["emb_dim"],
    mlp_dim=config["model"]["mlp_dim"],
    pos_emb_type=config["model"].get("pos_emb_type", "learned"),
    dtype=compute_dtype,
    param_dtype=jnp.float32
)

key = jax.random.PRNGKey(seed)
init_key, _ = jax.random.split(key)
dummy_input = jnp.zeros((1, config["model"]["context_len"]), dtype=jnp.int32)
params = model.init(init_key, dummy_input)["params"]

# Optimizer Schedule
total_steps = config["training"]["total_steps"]
warmup_steps = config["training"].get("warmup_steps", 100)
base_lr = config["training"]["learning_rate"]

warmup_fn = optax.linear_schedule(0.0, base_lr, warmup_steps)
cosine_fn = optax.cosine_decay_schedule(base_lr, max(1, total_steps - warmup_steps))
schedule_fn = optax.join_schedules([warmup_fn, cosine_fn], [warmup_steps])

tx = optax.chain(
    optax.clip_by_global_norm(config["training"].get("max_grad_norm", 1.0)),
    optax.adamw(learning_rate=schedule_fn, weight_decay=config["training"].get("weight_decay", 0.01))
)

state = CustomTrainState.create(apply_fn=model.apply, params=params, tx=tx)

# Multi-GPU Setup (Kaggle T4*2 or single GPU)
num_devices = jax.local_device_count()
batch_size = config["training"]["batch_size"]

if num_devices > 1:
    per_device_batch = batch_size // num_devices
    print(f"✅ Multi-GPU JAX pmap ACTIVE: Running on {num_devices} GPUs with per-device batch size {per_device_batch}")
    p_train_step = make_parallel_train_step()
    replicated_state = jutils.replicate(state)
else:
    per_device_batch = batch_size
    print(f"ℹ️ Single-device JAX ACTIVE: Running on {num_devices} device with batch size {batch_size}")

checkpoint_manager = CheckpointManager(config["training"]["checkpoint_dir"])
print("Initialization complete. Ready to launch training!")

In [ ]:
# Cell 5: Interactive Training Loop with On-The-Go Live Plotting
plotter = LivePlotter()
start_time = time.time()
cur_enabled = cur_cfg.get("enabled", True)

train_stream = sampler.stream_batches(
    batch_size=batch_size,
    get_probs_fn=cur_tracker.get_probabilities if cur_enabled else None,
    prefetch_size=2 if num_devices > 1 else 0
)

log_interval = config["training"].get("log_interval", 10)
eval_interval = config["training"].get("eval_interval", 100)
save_interval = config["training"].get("save_interval", 500)
target_accuracy = 0.95  # Target tolerant accuracy for early stopping in production

print(f"🚀 Starting Training Loop ({total_steps} steps total)...")

for step in range(1, total_steps + 1):
    batch = next(train_stream)
    
    if num_devices > 1:
        batch_reshaped = {
            "input_ids": batch["input_ids"].reshape(num_devices, per_device_batch, -1),
            "loss_mask": batch["loss_mask"].reshape(num_devices, per_device_batch, -1),
            "category_idx": batch["category_idx"].reshape(num_devices, per_device_batch)
        }
        replicated_state, loss_arr, seq_losses_arr = p_train_step(replicated_state, batch_reshaped)
        loss_val = float(loss_arr[0])
        seq_losses_val = np.array(seq_losses_arr[0]).flatten()
        cat_idxs = batch["category_idx"]
    else:
        state, loss_val, seq_losses_val = train_step(state, batch)
        cat_idxs = batch["category_idx"]
        
    if cur_enabled:
        for cat_idx, seq_loss in zip(cat_idxs, seq_losses_val):
            cur_tracker.update_train_loss(int(cat_idx), float(seq_loss))
            
    if cur_enabled and step % cur_tracker.update_interval == 0:
        cur_tracker.recompute_weights()
        
    if step % log_interval == 0 or step == 1:
        plotter.update_train(step, loss_val)
        
    if step % eval_interval == 0 or step == total_steps:
        current_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
        eval_metrics = evaluate_on_dataset(
            model=model,
            params=current_state.params,
            tokenizer=tokenizer,
            dataset=sampler.val_set,
            context_len=config["model"]["context_len"],
            epsilon=config["training"].get("accuracy_epsilon", 0.05)
        )
        
        val_loss = eval_metrics["overall/loss"]
        exact_match = eval_metrics["overall/exact_match"]
        tolerant_acc = eval_metrics["overall/tolerant_accuracy"]
        
        plotter.update_val(step, val_loss, exact_match, tolerant_acc)
        plotter.plot()
        
        if cur_enabled:
            val_losses_to_feed = {cat: eval_metrics[f"category_loss/{cat}"] for cat in sampler.categories if f"category_loss/{cat}" in eval_metrics}
            cur_tracker.update_val_losses(val_losses_to_feed)
            
        print(f"Step {step}/{total_steps} | Val Loss: {val_loss:.4f} | Exact Match: {exact_match*100:.2f}% | Tolerant Acc: {tolerant_acc*100:.2f}%")
        
        # Check early stopping condition for production training
        if not DEBUG_MODE and tolerant_acc >= target_accuracy and step >= 2000:
            print(f"🎯 Reached target accuracy threshold ({tolerant_acc*100:.2f}% >= {target_accuracy*100:.0f}%). Stopping training early!")
            break

    if step % save_interval == 0 or step == total_steps:
        current_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
        checkpoint_manager.save(step, current_state)
        
print("🎉 Training process finished successfully!")

In [ ]:
# Cell 6: Save Model and Dataset to Kaggle Working Directory
output_dir = "/kaggle/working" if os.path.exists("/kaggle") else "."
model_save_dir = os.path.join(output_dir, "saved_model")
dataset_save_dir = os.path.join(output_dir, "saved_dataset")

os.makedirs(model_save_dir, exist_ok=True)
os.makedirs(dataset_save_dir, exist_ok=True)

# 1. Save Final Model Checkpoint
final_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
checkpoint_manager.save(total_steps, final_state)
print(f"💾 Saved Model Checkpoints to: {config['training']['checkpoint_dir']}")

# 2. Export Generated Offline Dataset Shards
print("💾 Exporting generated dataset shards...")
sampler.dump_offline_dataset(
    output_dir=dataset_save_dir,
    num_examples=1000 if DEBUG_MODE else 10000,
    shard_size=1000 if DEBUG_MODE else 5000
)

# 3. Save Validation and Test Split JSONs
val_file = os.path.join(dataset_save_dir, "val_set.json")
test_file = os.path.join(dataset_save_dir, "test_set.json")

with open(val_file, "w") as f:
    json.dump(sampler.val_set, f, indent=2)

with open(test_file, "w") as f:
    json.dump(sampler.test_set, f, indent=2)

print(f"✅ Saved offline dataset shards to: {dataset_save_dir}")
print(f"   - Sharded JSONL files created: {len(os.listdir(dataset_save_dir))}")
print(f"   - Validation set saved: {val_file}")
print(f"   - Test set saved: {test_file}")
checkpoint_manager.close()